# Colab Runner: Swin SSL + Segmentation

This notebook runs your pipeline directly on Google Drive data.

It supports:
- quick smoke tests
- short sanity training runs
- full runs once configuration is verified

In [ ]:
# 1) Install dependencies
!pip -q install --upgrade transformers tqdm wandb


In [ ]:
# 2) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2b) Pull from GitHub (full git log: stash, fetch, fast-forward, file stats)
import os
from pathlib import Path

REPO = Path("/content/drive/My Drive/Payne_lab_swin_transformer")
if not REPO.is_dir():
    raise FileNotFoundError(f"Clone not found: {REPO} — mount Drive and fix the path.")
os.chdir(REPO)
print(REPO.resolve(), "\n")
!git status -sb
!git stash push -m "colab-local-before-pull"
!git pull origin feat/swin-upernet-training-pipeline

In [ ]:
# 2c) Weights & Biases login (run once per Colab runtime)
# Paste your W&B API key when prompted. Skip this cell to disable W&B logging.
import os
os.environ.setdefault('WANDB_PROJECT', 'payne-carbonate-segmentation')
import wandb
wandb.login()


In [ ]:
# 3) Configure repo path and dataset paths
import os
from pathlib import Path

# If your repo is in Drive, update this path if needed.
REPO_ROOT = Path("/content/drive/My Drive/Payne_lab_swin_transformer")

# Unlabeled data root used by SSL script (it recursively scans the 3 configured subfolders)
GDRIVE_ROOT = "/content/drive/My Drive/Petrographic images_ML work"

# Labeled image/mask dirs used by segmentation script
IMG_DIR = "/content/drive/My Drive/Petrographic images_ML work/labelled images_PS/labelledDataset_02032026/my_dataset/img"
MASK_DIR = "/content/drive/My Drive/Petrographic images_ML work/labelled images_PS/labelledDataset_02032026/my_dataset/masks_machine"

# Output folder for checkpoints/visualizations
OUT_ROOT = "/content/drive/My Drive/Petrographic images_ML work/model_outputs_221"

# For `!python ... $IMG_DIR` cells (bash reads these)
os.environ["REPO_ROOT"] = str(REPO_ROOT)
os.environ["GDRIVE_ROOT"] = GDRIVE_ROOT
os.environ["IMG_DIR"] = IMG_DIR
os.environ["MASK_DIR"] = MASK_DIR
os.environ["OUT_ROOT"] = OUT_ROOT

print("Repo exists:", REPO_ROOT.exists())
print("IMG dir exists:", Path(IMG_DIR).exists())
print("MASK dir exists:", Path(MASK_DIR).exists())

In [ ]:
# 3b) One-time: build a tarball of the unlabeled images on Drive
# This step is SLOW (it has to read every unlabeled image from Drive FUSE once)
# but it only ever needs to run ONCE per dataset version. After this exists,
# every future Colab session can use cell 3c to copy + extract the tarball in
# ~10 minutes instead of re-rsyncing ~14 GB of individual files for ~2 hours.
#
# If the tarball already exists on Drive, this cell is a no-op.
import os
from pathlib import Path

TARBALL_PATH = f"{GDRIVE_ROOT}/unlabeled_images.tar"
os.environ["TARBALL_PATH"] = TARBALL_PATH

if Path(TARBALL_PATH).exists():
    size_gb = Path(TARBALL_PATH).stat().st_size / (1024**3)
    print(f"[skip] tarball already exists ({size_gb:.2f} GB): {TARBALL_PATH}")
else:
    print(f"Building tarball at {TARBALL_PATH}")
    print("This will take a while (~30-60 min) but only needs to run once ever.")
    # -C changes to GDRIVE_ROOT before reading, so paths inside the tar are relative.
    !tar -cf "$TARBALL_PATH" \
        -C "$GDRIVE_ROOT" \
        "cretaceous thin sections" \
        "Permian-Triassic" \
        "TJ photomicrographs"
    print("Tarball built.")


In [ ]:
# 3c) Per-session: copy tarball from Drive to /content/ and extract
# Single-file Drive copy + local extract = ~10 minutes vs. ~2 hours of
# individual-file rsync. Idempotent: skips if data is already extracted.
import os
from pathlib import Path

LOCAL_UNLABELED_ROOT = "/content/unlabeled_local"
os.environ["LOCAL_UNLABELED_ROOT"] = LOCAL_UNLABELED_ROOT

subdirs = [
    "cretaceous thin sections",
    "Permian-Triassic",
    "TJ photomicrographs",
]

Path(LOCAL_UNLABELED_ROOT).mkdir(parents=True, exist_ok=True)

# Idempotency: check that ALL three subdirs exist locally with files.
all_present = all(
    (Path(LOCAL_UNLABELED_ROOT) / s).is_dir()
    and any((Path(LOCAL_UNLABELED_ROOT) / s).iterdir())
    for s in subdirs
)

if all_present:
    print(f"[skip] already extracted at {LOCAL_UNLABELED_ROOT}")
else:
    if not Path(TARBALL_PATH).exists():
        raise FileNotFoundError(
            f"Tarball not found at {TARBALL_PATH}. Run cell 3b first to build it."
        )
    print(f"Extracting {TARBALL_PATH} -> {LOCAL_UNLABELED_ROOT}")
    !tar -xf "$TARBALL_PATH" -C "$LOCAL_UNLABELED_ROOT"
    print("Extracted.")

n_local = !find "$LOCAL_UNLABELED_ROOT" -type f \( -iname '*.jpg' -o -iname '*.jpeg' -o -iname '*.png' -o -iname '*.tif' -o -iname '*.tiff' \) | wc -l
print(f"Local unlabeled images: {n_local[0].strip()}")


In [ ]:
# 4) Smoke tests (recommended first)
import os

os.chdir(REPO_ROOT)

# SSL smoke test: 1 epoch, 2 steps
!python -u code/model_training_pipeline/swin_ssl_pretrain_221.py \
  --gdrive_root "$GDRIVE_ROOT" \
  --epochs 1 \
  --batch_size 2 \
  --num_workers 2 \
  --max_steps_per_epoch 2 \
  --output_dir "$OUT_ROOT/ssl_smoke" \
  --amp

# Segmentation dataloader smoke test
!python -u code/model_training_pipeline/swin_training_pipeline_221.py \
  --img_dir "$IMG_DIR" \
  --mask_dir "$MASK_DIR" \
  --no_train

In [ ]:
# 5) Full SSL pretraining run (stage 1)
# 150 epochs of full SSL pretraining. Uses the local copy of unlabeled images
# for fast I/O, writes a log file you can tail in the next cell, and logs
# train loss + LR per epoch to W&B.
from pathlib import Path
import os

ssl_out = Path(f"{OUT_ROOT}/ssl_full")
ssl_out.mkdir(parents=True, exist_ok=True)
ssl_log = ssl_out / "ssl_run.log"
os.environ["SSL_OUT"] = str(ssl_out)
os.environ["SSL_LOG"] = str(ssl_log)

resume_path = ssl_out / "ssl_swinv2_last.pth"
resume_arg = f'--resume "{resume_path}"' if resume_path.exists() else ""
print("Resume mode:", "ON" if resume_arg else "OFF (starting fresh)")
print("SSL output:", ssl_out)
print("Log file:  ", ssl_log)

# Use local copy when available, fall back to Drive otherwise.
data_root = LOCAL_UNLABELED_ROOT if Path(LOCAL_UNLABELED_ROOT).is_dir() else GDRIVE_ROOT
print("Data root: ", data_root)

cmd = (
    "python -u code/model_training_pipeline/swin_ssl_pretrain_221.py "
    f"--gdrive_root \"{data_root}\" "
    "--epochs 150 "
    "--batch_size 8 "
    "--num_workers 4 "
    "--crop 512 "
    "--mask_patch 16 "
    "--mask_ratio 0.70 "
    "--save_recon_every 5 "
    "--num_recon_samples 2 "
    f"--output_dir \"{ssl_out}\" "
    "--amp "
    "--wandb_project payne-carbonate-segmentation "
    "--wandb_run_name ssl_local_mask070_e150 "
    f"{resume_arg} "
    f"> \"{ssl_log}\" 2>&1 &"
)

print("\nLaunching:\n", cmd, sep="")
get_ipython().system(cmd)
print("\nSSL launched in background. Tail the log with the next cell.")


In [ ]:
# 5b) Tail the live SSL log (stop with the cell-stop button — does NOT kill training)
!tail -f "$SSL_LOG"


In [ ]:
# 5c) Optional: clear reconstruction previews from previous SSL runs
# so the recon_previews folder only contains files from the current run.
from pathlib import Path
preview_dir = Path(f"{OUT_ROOT}/ssl_full/recon_previews")
if preview_dir.is_dir():
    pngs = list(preview_dir.glob("epoch_*.png"))
    for p in pngs:
        p.unlink()
    print(f"Removed {len(pngs)} stale previews from {preview_dir}")
else:
    print("No preview folder yet.")


In [ ]:
# 6) Full segmentation finetune run (stage 2)
# Class-weighted focal CE, scale-bar ignored, cosine LR with 5-epoch warmup, 150 epochs,
# SSL-init backbone, W&B logging on.
import os
os.chdir(REPO_ROOT)
!python -u code/model_training_pipeline/swin_training_pipeline_221.py \
  --img_dir "$IMG_DIR" \
  --mask_dir "$MASK_DIR" \
  --epochs 150 \
  --batch_size 2 \
  --crop 512 \
  --lr 3e-4 \
  --warmup_epochs 5 \
  --scheduler cosine \
  --auto_class_weights \
  --ignore_scale_bar \
  --loss_type focal \
  --focal_gamma 2.0 \
  --backbone_checkpoint "$OUT_ROOT/ssl_full/ssl_swinv2_best.pth" \
  --output_dir "$OUT_ROOT/seg_full_ssl_init" \
  --wandb_project payne-carbonate-segmentation \
  --wandb_run_name seg_focal_cw_warmup_e150


In [ ]:
# 7b) Confusion matrix on the validation split
# Loads the best segmentation checkpoint, rebuilds the same val split that
# training used (seed/val_frac), runs inference, and saves + displays:
#   - cm_raw.npy             (16x16 raw pixel counts)
#   - cm_row_normalized.png  (each row sums to 1; diagonal = recall)
#   - cm_col_normalized.png  (each col sums to 1; diagonal = precision)
import os, sys
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt
from torchvision.transforms.v2 import CenterCrop, Compose

# Make the training pipeline importable so we can reuse its helpers.
PIPE_DIR = REPO_ROOT / "code" / "model_training_pipeline"
if str(PIPE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPE_DIR))
from swin_training_pipeline_221 import (
    CLASS_NAMES,
    NUM_CLASSES,
    IGNORE_INDEX,
    BACKBONE_ID,
    CarbonateSegmentationDataset,
    confusion_matrix as cm_fn,
    get_model_semantic_segmentation,
)

# --- Config: must match the training run in cell 7 ---
SEG_OUT  = Path(f"{OUT_ROOT}/seg_full_ssl_init")
CKPT     = SEG_OUT / "best_upernet_swinv2.pth"
CROP     = 512
VAL_FRAC = 0.2
SEED     = 1337

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Rebuild the same val split training used (seed-based) ---
val_tf = Compose([CenterCrop((CROP, CROP))])
probe = CarbonateSegmentationDataset(
    root=".", img_dir=IMG_DIR, mask_dir=MASK_DIR,
    transforms=None, normalize=True, strict=True,
)
n = len(probe)
n_val = max(1, int(VAL_FRAC * n))
perm = torch.randperm(n, generator=torch.Generator().manual_seed(SEED))
val_idx = perm[:n_val].tolist()

val_full = CarbonateSegmentationDataset(
    root=".", img_dir=IMG_DIR, mask_dir=MASK_DIR,
    transforms=val_tf, normalize=True, strict=False, print_pair_count=False,
)
val_ds = Subset(val_full, val_idx)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=0,
                        pin_memory=torch.cuda.is_available())
print(f"Val samples: {len(val_ds)}")

# --- Load the trained model ---
model = get_model_semantic_segmentation(NUM_CLASSES, IGNORE_INDEX, BACKBONE_ID).to(device)
ckpt = torch.load(str(CKPT), map_location=device)
model.load_state_dict(ckpt["model_state"])
model.eval()
print(f"Loaded {CKPT}")

# --- Accumulate confusion matrix over val set ---
cm_total = torch.zeros(NUM_CLASSES, NUM_CLASSES, dtype=torch.float64)
with torch.no_grad():
    for imgs, labels in val_loader:
        imgs   = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        logits = model(pixel_values=imgs).logits
        logits = F.interpolate(logits, size=labels.shape[-2:], mode="bilinear", align_corners=False)
        preds  = logits.argmax(dim=1)
        cm_total += cm_fn(preds, labels, NUM_CLASSES, IGNORE_INDEX).cpu().double()

cm_np = cm_total.numpy()
np.save(SEG_OUT / "cm_raw.npy", cm_np)
print("Saved", SEG_OUT / "cm_raw.npy", "shape", cm_np.shape)

# --- Row- and column-normalized matrices ---
row_sums = cm_np.sum(axis=1, keepdims=True); row_sums[row_sums == 0] = 1
col_sums = cm_np.sum(axis=0, keepdims=True); col_sums[col_sums == 0] = 1
cm_row = cm_np / row_sums   # diagonal = recall
cm_col = cm_np / col_sums   # diagonal = precision

# --- Plot helper ---
def plot_cm(mat, title, out_path):
    fig, ax = plt.subplots(figsize=(11, 9))
    im = ax.imshow(mat, vmin=0, vmax=1, cmap="Blues")
    ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(CLASS_NAMES, fontsize=8)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Ground truth")
    ax.set_title(title)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            v = mat[i, j]
            if v > 0.005:
                ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                        fontsize=7, color="white" if v > 0.5 else "black")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("Saved", out_path)

plot_cm(cm_row, "Confusion matrix (row-normalized; diag = recall)",
        SEG_OUT / "cm_row_normalized.png")
plot_cm(cm_col, "Confusion matrix (col-normalized; diag = precision)",
        SEG_OUT / "cm_col_normalized.png")

# --- Per-class summary ---
diag = np.diag(cm_np)
recall    = diag / np.maximum(cm_np.sum(axis=1), 1)
precision = diag / np.maximum(cm_np.sum(axis=0), 1)
union     = cm_np.sum(axis=1) + cm_np.sum(axis=0) - diag
iou       = np.where(union > 0, diag / np.maximum(union, 1), np.nan)

print(f"\n{'class':<20}{'recall':>10}{'precision':>12}{'IoU':>10}")
for i, name in enumerate(CLASS_NAMES):
    iou_s = f"{iou[i]:.3f}" if np.isfinite(iou[i]) else "  nan"
    print(f"{name:<20}{recall[i]:>10.3f}{precision[i]:>12.3f}{iou_s:>10}")


In [ ]:
# 7) Show latest SSL reconstruction preview
# Previews are written under the same --output_dir you used, e.g. .../ssl_full/recon_previews.
from pathlib import Path
from IPython.display import Image as IPyImage, display

base = Path(OUT_ROOT)
candidates = [
    base / "ssl_full" / "recon_previews",
    base / "ssl_smoke" / "recon_previews",
] + list(base.rglob("recon_previews"))

seen = set()
preview_dir = None
for d in candidates:
    d = d.resolve() if d.exists() else None
    if d and d.is_dir() and d not in seen:
        seen.add(d)
        if any(d.glob("epoch_*.png")):
            preview_dir = d
            break

if preview_dir is None:
    for d in [base / "ssl_full" / "recon_previews", base / "ssl_smoke" / "recon_previews"]:
        if d.is_dir():
            preview_dir = d
            break

if preview_dir and preview_dir.exists():
    preview_files = sorted(preview_dir.glob("epoch_*.png"))
    if preview_files:
        print("Using:", preview_dir)
        print("Latest preview:", preview_files[-1])
        display(IPyImage(filename=str(preview_files[-1])))
    else:
        print("Folder exists but no epoch_*.png yet:", preview_dir, "\nRe-run SSL cell 5.")
else:
    print("No recon_previews folder found under", base)
    print("Run cell 5 first; SSL saves to <output_dir>/recon_previews/epoch_001.png by default.")